# Cell 1 — Publication-grade Version 2

# NA-HQKVE-IDS — Publication-Grade Experimental Notebook

This notebook extends the successful 4-qubit prototype into a controlled research-paper framework.

**Pipeline:** UNSW-NB15 → preprocessing → MI + XGBoost → denoising AE → 4/6 latent quantum features → Fixed Quantum Kernel / QKA-QSVC / VQC / LightGBM → calibrated stacking.

**Version 2 adds:** repeated seeds, 4Q vs 6Q, fixed-kernel vs QKA ablation, prototype-reduced QKA, low-data scaling, shot sensitivity, bounded adversarial latent-space stress tests, paired statistics, quantum-cost logging, uncertainty rejection, and IBM Runtime-ready hardware validation.

Start with `PROFILE="VALIDATE"`, then use `PAPER_CORE`, and only then `PAPER_FULL`.

In [ ]:
# Cell 2 — Install dependencies
!pip -q install -U "qiskit>=2.2" "qiskit-machine-learning>=0.9.0" "qiskit-aer>=0.17" "qiskit-ibm-runtime>=0.40" "scikit-learn>=1.4" "xgboost>=2.0" "lightgbm>=4.0" "imbalanced-learn>=0.12" torch pandas numpy scipy matplotlib joblib dill


In [ ]:
# Cell 3 — Imports and versions
import os,time,json,random,warnings,platform,math
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,RobustScaler,MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score,matthews_corrcoef,roc_auc_score,average_precision_score,brier_score_loss,confusion_matrix,classification_report
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import torch,torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import qiskit,qiskit_machine_learning,qiskit_aer
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map,real_amplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityStatevectorKernel,TrainableFidelityQuantumKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.optimizers import SPSA,COBYLA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import VQC
print('Python',platform.python_version()); print('Qiskit',qiskit.__version__); print('QML',qiskit_machine_learning.__version__); print('Aer',qiskit_aer.__version__); print('CUDA',torch.cuda.is_available())


In [ ]:
# Cell 4 — Experiment profiles
PROFILE='VALIDATE' # change to PAPER_CORE or PAPER_FULL
SEED_MASTER=42
random.seed(SEED_MASTER); np.random.seed(SEED_MASTER); torch.manual_seed(SEED_MASTER)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED_MASTER)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if PROFILE=='VALIDATE':
    SEEDS=[11]; QUBIT_LIST=[4]; Q_TRAIN_PER_CLASS_LIST=[30]; PROTOTYPES_PER_CLASS_LIST=[10]; QKA_MAXITER=4; VQC_MAXITER=15; AE_EPOCHS=20; Q_VAL_PER_CLASS=30; Q_TEST_PER_CLASS=60; SHOT_LIST=[256,1024]
elif PROFILE=='PAPER_CORE':
    SEEDS=[11,22,33]; QUBIT_LIST=[4,6]; Q_TRAIN_PER_CLASS_LIST=[50,100]; PROTOTYPES_PER_CLASS_LIST=[20,40]; QKA_MAXITER=8; VQC_MAXITER=35; AE_EPOCHS=50; Q_VAL_PER_CLASS=60; Q_TEST_PER_CLASS=120; SHOT_LIST=[1024,2048,4096]
else:
    SEEDS=[11,22,33,44,55]; QUBIT_LIST=[4,6]; Q_TRAIN_PER_CLASS_LIST=[50,100,250]; PROTOTYPES_PER_CLASS_LIST=[20,50,100]; QKA_MAXITER=12; VQC_MAXITER=60; AE_EPOCHS=80; Q_VAL_PER_CLASS=100; Q_TEST_PER_CLASS=250; SHOT_LIST=[1024,2048,4096]
TOP_MI=25; TOP_FINAL=12
RUN_QKA=True; RUN_VQC=True; RUN_PROTOTYPE_QKA=True; RUN_SHOT_KERNEL=True; RUN_ADVERSARIAL_STRESS=True; RUN_IBM_HARDWARE=False
print(PROFILE,DEVICE,SEEDS,QUBIT_LIST,Q_TRAIN_PER_CLASS_LIST)


In [ ]:
# Cell 5 — Upload UNSW-NB15
from google.colab import files
print('Upload UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv')
files.upload(); csvs=list(Path('/content').glob('*.csv'))
def locate(keys):
    for p in csvs:
        n=p.name.lower()
        if any(k in n for k in keys): return str(p)
TRAIN_PATH=locate(['training-set','training']); TEST_PATH=locate(['testing-set','testing'])
if not TRAIN_PATH or not TEST_PATH: raise FileNotFoundError('Training/testing files not detected.')
print(TRAIN_PATH,TEST_PATH)


In [ ]:
# Cell 6 — Load and clean binary IDS data
train_df=pd.read_csv(TRAIN_PATH); test_df=pd.read_csv(TEST_PATH)
for df in (train_df,test_df): df.replace([np.inf,-np.inf],np.nan,inplace=True)
DROP=['id','attack_cat']
X_train_raw=train_df.drop(columns=[c for c in DROP+['label'] if c in train_df]).copy(); X_test_raw=test_df.drop(columns=[c for c in DROP+['label'] if c in test_df]).copy(); X_test_raw=X_test_raw.reindex(columns=X_train_raw.columns)
y_train=train_df['label'].astype(int).to_numpy(); y_test=test_df['label'].astype(int).to_numpy()
cat_cols=X_train_raw.select_dtypes(include=['object','category']).columns.tolist(); num_cols=[c for c in X_train_raw.columns if c not in cat_cols]
X_train_raw[num_cols]=X_train_raw[num_cols].apply(pd.to_numeric,errors='coerce'); X_test_raw[num_cols]=X_test_raw[num_cols].apply(pd.to_numeric,errors='coerce')
med=X_train_raw[num_cols].median(); X_train_raw[num_cols]=X_train_raw[num_cols].fillna(med); X_test_raw[num_cols]=X_test_raw[num_cols].fillna(med)
for c in cat_cols: X_train_raw[c]=X_train_raw[c].fillna('MISSING').astype(str); X_test_raw[c]=X_test_raw[c].fillna('MISSING').astype(str)
print(X_train_raw.shape,np.bincount(y_train)); print(X_test_raw.shape,np.bincount(y_test)); print(cat_cols)


In [ ]:
# Cell 7 — Leakage-safe encoding and scaling
preprocessor=ColumnTransformer([('num',RobustScaler(),num_cols),('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat_cols)],verbose_feature_names_out=False)
Xtr=preprocessor.fit_transform(X_train_raw); Xte=preprocessor.transform(X_test_raw); names=preprocessor.get_feature_names_out(); Xtr=pd.DataFrame(Xtr,columns=names); Xte=pd.DataFrame(Xte,columns=names); print(Xtr.shape,Xte.shape)


In [ ]:
# Cell 8 — Mutual information screening
mi=mutual_info_classif(Xtr,y_train,random_state=SEED_MASTER); mi_s=pd.Series(mi,index=Xtr.columns).sort_values(ascending=False); mi_feats=mi_s.head(min(TOP_MI,len(mi_s))).index.tolist(); display(mi_s.head(TOP_MI).to_frame('MI')); Xtr_mi=Xtr[mi_feats]; Xte_mi=Xte[mi_feats]


In [ ]:
# Cell 9 — XGBoost selection to 12 features
params=dict(n_estimators=250,max_depth=5,learning_rate=.05,subsample=.85,colsample_bytree=.85,tree_method='hist',eval_metric='logloss',random_state=SEED_MASTER,n_jobs=-1)
if torch.cuda.is_available(): params['device']='cuda'
try: fs_model=XGBClassifier(**params).fit(Xtr_mi,y_train)
except Exception: params.pop('device',None); fs_model=XGBClassifier(**params).fit(Xtr_mi,y_train)
imp=pd.Series(fs_model.feature_importances_,index=mi_feats).sort_values(ascending=False); selected=imp.head(TOP_FINAL).index.tolist(); display(imp.head(TOP_FINAL).to_frame('importance')); Xtr_sel=Xtr_mi[selected].to_numpy(np.float32); Xte_sel=Xte_mi[selected].to_numpy(np.float32)


In [ ]:
# Cell 10 — Define denoising autoencoder
ae_scaler=MinMaxScaler((0,1)); Xtr_ae=ae_scaler.fit_transform(Xtr_sel).astype(np.float32); Xte_ae=ae_scaler.transform(Xte_sel).astype(np.float32)
class DAE(nn.Module):
    def __init__(self,input_dim,latent_dim=6):
        super().__init__(); self.encoder=nn.Sequential(nn.Linear(input_dim,10),nn.ReLU(),nn.Linear(10,8),nn.ReLU(),nn.Linear(8,latent_dim)); self.decoder=nn.Sequential(nn.Linear(latent_dim,8),nn.ReLU(),nn.Linear(8,10),nn.ReLU(),nn.Linear(10,input_dim),nn.Sigmoid())
    def forward(self,x): return self.decoder(self.encoder(x))
MAX_LATENT=max(QUBIT_LIST); ae=DAE(Xtr_ae.shape[1],MAX_LATENT).to(DEVICE); print(ae)


In [ ]:
# Cell 11 — Train denoising autoencoder
Xa_fit,Xa_val=train_test_split(Xtr_ae,test_size=.15,random_state=SEED_MASTER,stratify=y_train); loader=DataLoader(TensorDataset(torch.tensor(Xa_fit,dtype=torch.float32)),batch_size=512 if torch.cuda.is_available() else 256,shuffle=True); opt=torch.optim.Adam(ae.parameters(),lr=1e-3,weight_decay=1e-5); lossfn=nn.MSELoss(); val_t=torch.tensor(Xa_val,dtype=torch.float32,device=DEVICE)
best=np.inf; best_state=None; hist=[]
for epoch in range(1,AE_EPOCHS+1):
    ae.train(); ls=[]
    for (b,) in loader:
        b=b.to(DEVICE); noisy=torch.clamp(b+.05*torch.randn_like(b),0,1); opt.zero_grad(); rec=ae(noisy); loss=lossfn(rec,b); loss.backward(); opt.step(); ls.append(loss.item())
    ae.eval()
    with torch.no_grad(): vl=lossfn(ae(val_t),val_t).item()
    hist.append([epoch,float(np.mean(ls)),vl])
    if vl<best: best=vl; best_state={k:v.detach().cpu().clone() for k,v in ae.state_dict().items()}
    if epoch==1 or epoch%10==0 or epoch==AE_EPOCHS: print(epoch,hist[-1])
ae.load_state_dict(best_state); hist=pd.DataFrame(hist,columns=['epoch','train_mse','val_mse']); hist.plot(x='epoch',y=['train_mse','val_mse'],figsize=(8,4),title='DAE learning'); plt.show()


In [ ]:
# Cell 12 — Encode latent features and reconstruction error
def encode_err(X):
    ae.eval(); x=torch.tensor(X,dtype=torch.float32,device=DEVICE)
    with torch.no_grad(): z=ae.encoder(x); rec=ae.decoder(z); e=torch.mean((rec-x)**2,dim=1)
    return z.cpu().numpy(),e.cpu().numpy()
Ztr6,Etr=encode_err(Xtr_ae); Zte6,Ete=encode_err(Xte_ae); err_scaler=MinMaxScaler().fit(Etr.reshape(-1,1)); print(Ztr6.shape,Zte6.shape)


In [ ]:
# Cell 13 — Balanced sampling utilities
def balanced_indices(y,n_per_class,seed):
    rng=np.random.default_rng(seed); parts=[]
    for cls in np.unique(y):
        ids=np.flatnonzero(y==cls); parts.append(rng.choice(ids,size=min(n_per_class,len(ids)),replace=False))
    out=np.concatenate(parts); rng.shuffle(out); return out
def disjoint_balanced_train_val(y,ntrain,nval,seed):
    rng=np.random.default_rng(seed); tr=[]; va=[]
    for cls in np.unique(y):
        ids=np.flatnonzero(y==cls).copy(); rng.shuffle(ids); tr.extend(ids[:min(ntrain,len(ids))]); left=ids[min(ntrain,len(ids)):]; va.extend(left[:min(nval,len(left))])
    tr=np.array(tr); va=np.array(va); rng.shuffle(tr); rng.shuffle(va); return tr,va
def scale_latent(Ztr,Zte,nq):
    s=MinMaxScaler((0,np.pi)); a=s.fit_transform(Ztr[:,:nq]); b=s.transform(Zte[:,:nq]); return np.clip(a,0,np.pi),np.clip(b,0,np.pi),s


In [ ]:
# Cell 14 — Metrics, logging and prototypes
RESULTS=[]
def metric_dict(y,p):
    pred=(np.asarray(p)>=.5).astype(int); return dict(accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),f1=f1_score(y,pred,zero_division=0),balanced_accuracy=balanced_accuracy_score(y,pred),mcc=matthews_corrcoef(y,pred),roc_auc=roc_auc_score(y,p),pr_auc=average_precision_score(y,p),brier=brier_score_loss(y,p))
def log_result(model,seed,nq,ntrain,nproto,y,p,seconds=np.nan,shots=0,scenario='clean',notes=''):
    row=dict(model=model,seed=seed,qubits=nq,train_per_class=ntrain,prototypes_per_class=nproto,shots=shots,scenario=scenario,seconds=seconds,notes=notes); row.update(metric_dict(y,p)); RESULTS.append(row)
def nearest_prototypes(X,y,m_per_class,seed):
    selected=[]
    for cls in np.unique(y):
        ids=np.flatnonzero(y==cls); Xc=X[ids]; m=min(m_per_class,len(Xc)); km=KMeans(n_clusters=m,random_state=seed,n_init=10).fit(Xc)
        for c in km.cluster_centers_: selected.append(ids[np.argmin(np.sum((Xc-c)**2,axis=1))])
    return np.unique(selected)


# Cell 15 — Ablation design

For matched seeds, qubits and training sizes, evaluate **LightGBM**, **Fixed-QK**, **QKA-QSVC**, **Prototype-QKA**, **VQC**, and **Stacked ensemble**. This identifies where any gain actually comes from.

In [ ]:
# Cell 16 — LightGBM helper
def train_lgbm(Ztr,ytr,Zval,Ztest,seed):
    m=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=31,class_weight='balanced',random_state=seed,n_jobs=-1,verbosity=-1); t=time.time(); m.fit(Ztr,ytr); sec=time.time()-t; return m,m.predict_proba(Zval)[:,1],m.predict_proba(Ztest)[:,1],sec


In [ ]:
# Cell 17 — Fixed quantum kernel helper
def fixed_qkernel_probs(Xtrain,ytrain,Xval,Xtest,nq):
    fmap=zz_feature_map(feature_dimension=nq,reps=1,entanglement='linear'); kernel=FidelityStatevectorKernel(feature_map=fmap,enforce_psd=True); model=QSVC(quantum_kernel=kernel,probability=True,class_weight='balanced'); t=time.time(); model.fit(Xtrain,ytrain); sec=time.time()-t; return model,model.predict_proba(Xval)[:,1],model.predict_proba(Xtest)[:,1],sec


In [ ]:
# Cell 18 — QKA-QSVC helper
def qka_probs(Xtrain,ytrain,Xval,Xtest,nq,seed,maxiter):
    theta=ParameterVector(f'theta_{seed}_{nq}',nq); layer=QuantumCircuit(nq)
    for i in range(nq): layer.ry(theta[i],i)
    fmap=layer.compose(zz_feature_map(feature_dimension=nq,reps=1,entanglement='linear')); sampler=StatevectorSampler(seed=seed); fidelity=ComputeUncompute(sampler=sampler); kernel=TrainableFidelityQuantumKernel(fidelity=fidelity,feature_map=fmap,training_parameters=theta); trainer=QuantumKernelTrainer(quantum_kernel=kernel,loss='svc_loss',optimizer=SPSA(maxiter=maxiter,learning_rate=.05,perturbation=.05),initial_point=np.full(nq,np.pi/2)); t=time.time(); result=trainer.fit(Xtrain,ytrain); trained=result.quantum_kernel; svc=QSVC(quantum_kernel=trained,probability=True,class_weight='balanced'); svc.fit(Xtrain,ytrain); sec=time.time()-t; return svc,trained,result,svc.predict_proba(Xval)[:,1],svc.predict_proba(Xtest)[:,1],sec


In [ ]:
# Cell 19 — VQC helper
def vqc_probs(Xtrain,ytrain,Xval,Xtest,nq,seed,maxiter):
    fmap=zz_feature_map(feature_dimension=nq,reps=1,entanglement='linear'); ansatz=real_amplitudes(num_qubits=nq,reps=1,entanglement='linear'); sampler=StatevectorSampler(seed=seed); values=[]; model=VQC(sampler=sampler,feature_map=fmap,ansatz=ansatz,optimizer=COBYLA(maxiter=maxiter),callback=lambda w,o: values.append(float(o))); t=time.time(); model.fit(Xtrain,ytrain); sec=time.time()-t; return model,model.predict_proba(Xval)[:,1],model.predict_proba(Xtest)[:,1],sec,values


In [ ]:
# Cell 20 — Run core publication grid
CACHE={}; test_idx=balanced_indices(y_test,Q_TEST_PER_CLASS,SEED_MASTER+999)
for nq in QUBIT_LIST:
    Ztr_q,Zte_q,q_scaler=scale_latent(Ztr6,Zte6,nq)
    for ntrain in Q_TRAIN_PER_CLASS_LIST:
        for seed in SEEDS:
            print(f'=== nq={nq} train/class={ntrain} seed={seed} ==='); tr_idx,val_idx=disjoint_balanced_train_val(y_train,ntrain,Q_VAL_PER_CLASS,seed); Xqtr,yqtr=Ztr_q[tr_idx],y_train[tr_idx]; Xqv,yqv=Ztr_q[val_idx],y_train[val_idx]; Xqt,yqt=Zte_q[test_idx],y_test[test_idx]
            lgb,pL_v,pL_t,tL=train_lgbm(Ztr6[:,:nq],y_train,Ztr6[val_idx,:nq],Zte6[test_idx,:nq],seed); log_result('LightGBM',seed,nq,ntrain,0,yqt,pL_t,tL)
            fixed,pF_v,pF_t,tF=fixed_qkernel_probs(Xqtr,yqtr,Xqv,Xqt,nq); log_result('Fixed_QK',seed,nq,ntrain,0,yqt,pF_t,tF)
            qka=kernel=qka_res=None; pQ_v=pQ_t=None
            if RUN_QKA:
                qka,kernel,qka_res,pQ_v,pQ_t,tQ=qka_probs(Xqtr,yqtr,Xqv,Xqt,nq,seed,QKA_MAXITER); log_result('QKA_QSVC',seed,nq,ntrain,0,yqt,pQ_t,tQ)
            vqc=None; pV_v=pV_t=None
            if RUN_VQC:
                vqc,pV_v,pV_t,tV,vhist=vqc_probs(Xqtr,yqtr,Xqv,Xqt,nq,seed,VQC_MAXITER); log_result('VQC',seed,nq,ntrain,0,yqt,pV_t,tV)
            stack=None
            if pQ_v is not None and pV_v is not None:
                ev=np.clip(err_scaler.transform(Etr[val_idx,None]).ravel(),0,1); et=np.clip(err_scaler.transform(Ete[test_idx,None]).ravel(),0,1); Sval=np.column_stack([pQ_v,pV_v,pL_v,ev]); Stest=np.column_stack([pQ_t,pV_t,pL_t,et]); stack=LogisticRegression(class_weight='balanced',random_state=seed,max_iter=1000).fit(Sval,yqv); pS=stack.predict_proba(Stest)[:,1]; log_result('Stack_QKA_VQC_LGBM',seed,nq,ntrain,0,yqt,pS,0)
            CACHE[(nq,ntrain,seed)]=dict(tr_idx=tr_idx,val_idx=val_idx,test_idx=test_idx,scaler=q_scaler,Xqtr=Xqtr,yqtr=yqtr,Xqv=Xqv,yqv=yqv,Xqt=Xqt,yqt=yqt,fixed=fixed,qka=qka,qka_kernel=kernel,qka_result=qka_res,vqc=vqc,lgbm=lgb,stacker=stack,probs=dict(fixed_val=pF_v,fixed_test=pF_t,qka_val=pQ_v,qka_test=pQ_t,vqc_val=pV_v,vqc_test=pV_t,lgbm_val=pL_v,lgbm_test=pL_t))
results_df=pd.DataFrame(RESULTS); display(results_df)


In [ ]:
# Cell 21 — Prototype-reduced QKA
if RUN_PROTOTYPE_QKA and RUN_QKA:
    for nq in QUBIT_LIST:
        for ntrain in Q_TRAIN_PER_CLASS_LIST:
            for seed in SEEDS:
                b=CACHE[(nq,ntrain,seed)]
                for m in PROTOTYPES_PER_CLASS_LIST:
                    if m>=ntrain: continue
                    pidx=nearest_prototypes(b['Xqtr'],b['yqtr'],m,seed); Xp,yp=b['Xqtr'][pidx],b['yqtr'][pidx]; print('Prototype QKA',nq,ntrain,seed,len(pidx)); _,_,_,_,ptest,sec=qka_probs(Xp,yp,b['Xqv'],b['Xqt'],nq,seed,QKA_MAXITER); log_result('Prototype_QKA',seed,nq,ntrain,m,b['yqt'],ptest,sec,notes=f'{len(pidx)} total prototypes')
results_df=pd.DataFrame(RESULTS); display(results_df[results_df.model=='Prototype_QKA'] if 'Prototype_QKA' in results_df.model.values else pd.DataFrame())


# Cell 22 — Shot sensitivity

Use `FidelityStatevectorKernel(shots=...)` to separate finite-shot sampling effects from hardware-specific noise. Report exact statevector results alongside 1024/2048/4096-shot results.

In [ ]:
# Cell 23 — Run shot-based fixed-kernel sensitivity
if RUN_SHOT_KERNEL:
    ntrain=max(Q_TRAIN_PER_CLASS_LIST); seed=SEEDS[0]
    for nq in QUBIT_LIST:
        b=CACHE[(nq,ntrain,seed)]; fmap=zz_feature_map(feature_dimension=nq,reps=1,entanglement='linear')
        for shots in SHOT_LIST:
            print('shot kernel',nq,shots); k=FidelityStatevectorKernel(feature_map=fmap,shots=shots,enforce_psd=True); svc=QSVC(quantum_kernel=k,probability=True,class_weight='balanced'); t=time.time(); svc.fit(b['Xqtr'],b['yqtr']); p=svc.predict_proba(b['Xqt'])[:,1]; log_result('Fixed_QK_Shot',seed,nq,ntrain,0,b['yqt'],p,time.time()-t,shots=shots)
results_df=pd.DataFrame(RESULTS)


# Cell 24 — Adversarial stress-test scope

This is a bounded continuous latent-space robustness test, not a claim of fully traffic-valid raw-feature FGSM/PGD. A later paper extension should impose explicit protocol/domain constraints on raw network features.

In [ ]:
# Cell 25 — Finite-difference latent attack helper
def finite_difference_attack(model_prob,X,y,eps=.08,steps=1):
    Xa=X.copy(); x0=X.copy(); delta=max(eps/4,1e-3); alpha=eps/max(steps,1)
    def loss(p,y): p=np.clip(p,1e-6,1-1e-6); return -(y*np.log(p)+(1-y)*np.log(1-p))
    for _ in range(steps):
        grad=np.zeros_like(Xa)
        for j in range(Xa.shape[1]):
            xp=Xa.copy(); xm=Xa.copy(); xp[:,j]=np.clip(xp[:,j]+delta,0,np.pi); xm[:,j]=np.clip(xm[:,j]-delta,0,np.pi); grad[:,j]=(loss(model_prob(xp),y)-loss(model_prob(xm),y))/(2*delta)
        Xa=np.clip(Xa+alpha*np.sign(grad),x0-eps,x0+eps); Xa=np.clip(Xa,0,np.pi)
    return Xa


In [ ]:
# Cell 26 — Run bounded adversarial stress test
if RUN_ADVERSARIAL_STRESS and RUN_QKA:
    nq=max(QUBIT_LIST); ntrain=max(Q_TRAIN_PER_CLASS_LIST); seed=SEEDS[0]; b=CACHE[(nq,ntrain,seed)]; nadv=min(40,len(b['yqt'])); X0=b['Xqt'][:nadv].copy(); y0=b['yqt'][:nadv].copy(); probfun=lambda X:b['qka'].predict_proba(X)[:,1]
    for attack_name,eps,steps in [('FGSM_like',.08,1),('PGD_like',.08,3)]:
        print(attack_name); Xa=finite_difference_attack(probfun,X0,y0,eps,steps); p=probfun(Xa); log_result('QKA_QSVC',seed,nq,ntrain,0,y0,p,scenario=attack_name,notes=f'bounded latent eps={eps}; steps={steps}')
results_df=pd.DataFrame(RESULTS)


In [ ]:
# Cell 27 — Aggregate publication results
results_df=pd.DataFrame(RESULTS); summary=(results_df[results_df.scenario=='clean'].groupby(['model','qubits','train_per_class','prototypes_per_class','shots'],as_index=False).agg(f1_mean=('f1','mean'),f1_sd=('f1','std'),mcc_mean=('mcc','mean'),mcc_sd=('mcc','std'),pr_auc_mean=('pr_auc','mean'),pr_auc_sd=('pr_auc','std'),recall_mean=('recall','mean'),recall_sd=('recall','std'),brier_mean=('brier','mean'),seconds_mean=('seconds','mean'))); display(summary.sort_values('f1_mean',ascending=False))


In [ ]:
# Cell 28 — Resource-performance plots
clean=results_df[results_df.scenario=='clean'].copy(); plt.figure(figsize=(8,5))
for model,g in clean[clean.shots==0].groupby('model'):
    gg=g.groupby('qubits')['f1'].mean(); plt.plot(gg.index,gg.values,marker='o',label=model)
plt.xlabel('Qubits'); plt.ylabel('Mean F1'); plt.title('F1 versus qubit count'); plt.legend(); plt.show(); plt.figure(figsize=(8,5))
for model,g in clean[(clean.shots==0)&(clean.prototypes_per_class==0)].groupby('model'):
    gg=g.groupby('train_per_class')['f1'].mean(); plt.plot(gg.index,gg.values,marker='o',label=model)
plt.xlabel('Quantum training samples per class'); plt.ylabel('Mean F1'); plt.title('Low-data scaling'); plt.legend(); plt.show()


In [ ]:
# Cell 29 — Paired Wilcoxon tests
x=results_df[(results_df.scenario=='clean')&(results_df.shots==0)&(results_df.prototypes_per_class==0)]; paired=x.pivot_table(index=['seed','qubits','train_per_class'],columns='model',values='f1',aggfunc='first'); display(paired); tests=[]
for a,b in [('QKA_QSVC','Fixed_QK'),('Stack_QKA_VQC_LGBM','LightGBM'),('VQC','LightGBM')]:
    if a in paired.columns and b in paired.columns:
        z=paired[[a,b]].dropna()
        if len(z)>=2:
            stat,p=wilcoxon(z[a],z[b],zero_method='wilcox'); d=(z[a]-z[b]).to_numpy(); tests.append(dict(model_A=a,model_B=b,n_pairs=len(z),mean_difference=d.mean(),median_difference=np.median(d),wilcoxon_stat=stat,p_value=p))
stats_df=pd.DataFrame(tests); display(stats_df)


In [ ]:
# Cell 30 — Holm correction
def holm_adjust(pvals):
    p=np.asarray(pvals,float); m=len(p); order=np.argsort(p); adj=np.empty(m); running=0
    for rank,idx in enumerate(order): running=max(running,(m-rank)*p[idx]); adj[idx]=min(running,1.0)
    return adj
if len(stats_df): stats_df['p_holm']=holm_adjust(stats_df.p_value.values)
display(stats_df)


In [ ]:
# Cell 31 — Uncertainty rejection analysis
if RUN_QKA and RUN_VQC:
    nq=max(QUBIT_LIST); ntrain=max(Q_TRAIN_PER_CLASS_LIST); seed=SEEDS[0]; b=CACHE[(nq,ntrain,seed)]; ev=np.clip(err_scaler.transform(Etr[b['val_idx'],None]).ravel(),0,1); et=np.clip(err_scaler.transform(Ete[b['test_idx'],None]).ravel(),0,1); Sval=np.column_stack([b['probs']['qka_val'],b['probs']['vqc_val'],b['probs']['lgbm_val'],ev]); Stest=np.column_stack([b['probs']['qka_test'],b['probs']['vqc_test'],b['probs']['lgbm_test'],et]); st=LogisticRegression(class_weight='balanced',max_iter=1000).fit(Sval,b['yqv']); pf=st.predict_proba(Stest)[:,1]; rows=[]
    for margin in [.05,.10,.15,.20]:
        keep=(pf<.5-margin)|(pf>.5+margin)
        if keep.sum(): rows.append({'margin':margin,'coverage':keep.mean(),**metric_dict(b['yqt'][keep],pf[keep])})
    rejection_df=pd.DataFrame(rows); display(rejection_df)


# Cell 32 — IBM hardware validation

Run hardware only after simulator results are frozen. Use one selected 4Q and 6Q circuit family, representative samples, Runtime V2 primitives, and record backend, transpiled depth, two-qubit gates, shots, job ID and calibration context.

In [ ]:
# Cell 33 — IBM Runtime connection template
if RUN_IBM_HARDWARE:
    from google.colab import userdata
    from qiskit_ibm_runtime import QiskitRuntimeService,SamplerV2
    token=userdata.get('IBM_QUANTUM_TOKEN')
    if token is None: raise ValueError('Store IBM_QUANTUM_TOKEN in Colab Secrets.')
    service=QiskitRuntimeService(channel='ibm_quantum_platform',token=token); nq=max(QUBIT_LIST); backend=service.least_busy(operational=True,simulator=False,min_num_qubits=nq); print(backend.name)
else: print('RUN_IBM_HARDWARE=False')


In [ ]:
# Cell 34 — Transpile representative circuit and log resources
if RUN_IBM_HARDWARE:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    nq=max(QUBIT_LIST); fmap=zz_feature_map(feature_dimension=nq,reps=1,entanglement='linear'); pm=generate_preset_pass_manager(backend=backend,optimization_level=3); isa=pm.run(fmap); ops=isa.count_ops(); resource_row={'backend':backend.name,'logical_qubits':nq,'original_depth':fmap.depth(),'transpiled_depth':isa.depth(),'size':isa.size(),'two_qubit_gates':sum(v for k,v in ops.items() if k in ['cx','cz','ecr']),'operations':dict(ops)}; display(pd.DataFrame([resource_row]))


In [ ]:
# Cell 35 — SamplerV2 representative execution
if RUN_IBM_HARDWARE:
    from qiskit_ibm_runtime import SamplerV2
    sampler=SamplerV2(mode=backend); key=(max(QUBIT_LIST),max(Q_TRAIN_PER_CLASS_LIST),SEEDS[0]); sample=CACHE[key]['Xqt'][0]; bound=fmap.assign_parameters(sample); bound_isa=pm.run(bound); job=sampler.run([bound_isa],shots=1024); print('Job ID',job.job_id()); print(job.result())
else: print('Hardware execution skipped.')


In [ ]:
# Cell 36 — Kernel cost estimator
def kernel_pair_count(n_train,n_test=0): return n_train*(n_train+1)//2+n_train*n_test
cost_rows=[]
for nq in QUBIT_LIST:
    for nt in Q_TRAIN_PER_CLASS_LIST:
        ntrain=2*nt; ntest=2*Q_TEST_PER_CLASS; cost_rows.append({'qubits':nq,'design':'full_kernel','train_samples':ntrain,'test_samples':ntest,'approx_unique_pair_evaluations':kernel_pair_count(ntrain,ntest)})
        for m in PROTOTYPES_PER_CLASS_LIST:
            np_=2*min(m,nt); cost_rows.append({'qubits':nq,'design':f'prototype_{m}_per_class','train_samples':np_,'test_samples':ntest,'approx_unique_pair_evaluations':kernel_pair_count(np_,ntest)})
cost_df=pd.DataFrame(cost_rows); display(cost_df)


In [ ]:
# Cell 37 — Paper-ready tables
table_core=results_df[(results_df.scenario=='clean')&(results_df.shots==0)].groupby(['model','qubits','train_per_class','prototypes_per_class'])[['accuracy','recall','f1','mcc','pr_auc','brier','seconds']].agg(['mean','std']); display(table_core); table_robust=results_df[results_df.scenario!='clean'][['model','qubits','train_per_class','scenario','accuracy','recall','f1','mcc','pr_auc']]; display(table_robust); table_shots=results_df[results_df.shots>0][['model','qubits','train_per_class','shots','accuracy','recall','f1','mcc','pr_auc','seconds']]; display(table_shots)


In [ ]:
# Cell 38 — Save research artifacts
import joblib,shutil
OUT=Path('/content/NA_HQKVE_IDS_V2'); OUT.mkdir(exist_ok=True); results_df.to_csv(OUT/'results_long.csv',index=False); summary.to_csv(OUT/'summary.csv',index=False); stats_df.to_csv(OUT/'statistical_tests.csv',index=False); cost_df.to_csv(OUT/'kernel_cost_estimates.csv',index=False); hist.to_csv(OUT/'dae_history.csv',index=False); pd.DataFrame({'feature':selected,'importance':[imp[x] for x in selected]}).to_csv(OUT/'selected_features.csv',index=False); joblib.dump(preprocessor,OUT/'preprocessor.joblib'); joblib.dump(ae_scaler,OUT/'ae_scaler.joblib'); joblib.dump(err_scaler,OUT/'error_scaler.joblib'); torch.save(ae.state_dict(),OUT/'dae_6latent.pt'); config={'profile':PROFILE,'seeds':SEEDS,'qubits':QUBIT_LIST,'q_train_per_class':Q_TRAIN_PER_CLASS_LIST,'q_val_per_class':Q_VAL_PER_CLASS,'q_test_per_class':Q_TEST_PER_CLASS,'prototypes_per_class':PROTOTYPES_PER_CLASS_LIST,'qka_maxiter':QKA_MAXITER,'vqc_maxiter':VQC_MAXITER,'shots':SHOT_LIST,'selected_features':selected,'versions':{'qiskit':qiskit.__version__,'qiskit_machine_learning':qiskit_machine_learning.__version__,'qiskit_aer':qiskit_aer.__version__,'torch':torch.__version__}}; json.dump(config,open(OUT/'experiment_config.json','w'),indent=2); print(OUT); [print(' -',p.name) for p in sorted(OUT.iterdir())]


In [ ]:
# Cell 39 — Zip and download results
from google.colab import files
zip_path=shutil.make_archive('/content/NA_HQKVE_IDS_V2','zip',str(OUT)); print(zip_path); files.download(zip_path)


# Cell 40 — Recommended paper execution order

1. Validate the entire notebook with `PROFILE="VALIDATE"`.
2. Run `PAPER_CORE` and inspect all ablations.
3. Use `PAPER_FULL` only after fixing the final configuration.
4. Do not interpret one accuracy result as quantum advantage.
5. Report mean ± SD, matched-seed statistics, effect direction, calibration, runtime and kernel-cost estimates.
6. Run IBM hardware only on the final selected circuit/prototype configuration.
7. Add raw-feature traffic-valid adversarial constraints before describing FGSM/PGD as realistic network attacks.
8. External validation on CIC-IDS2017 should be the next notebook/version.